## Setup

# CFD Data Processing - Part 2: Convergence Analysis (OPTIONAL)

This notebook performs convergence analysis to find optimal data trim points.
**This step is optional** - if you skip this, Part 3 will use fixed iterations (150 points).

Creates convergence plots and saves optimized trim values for use in Part 3 (Excel outputs).

In [9]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
from collections import defaultdict
import glob
import time

## Load Processed Data

In [10]:
# Automatically find the pickle file generated by Part 1
# This searches for processed_data.pkl starting from user's home directory

# Start search from a higher directory to find the Thesis folder
# Try multiple search locations to find the pickle file
search_roots = [
    r"C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis",  # Direct path to Thesis folder
    os.path.expanduser("~"),  # User home directory
]

pickle_files = []
for root in search_roots:
    if os.path.exists(root):
        search_pattern = os.path.join(root, "**", "processed_data.pkl")
        found = glob.glob(search_pattern, recursive=True)
        pickle_files.extend(found)
        if found:
            break  # Stop searching once we find files

# Filter to get the most recently modified pickle file
if pickle_files:
    pickle_file = max(pickle_files, key=os.path.getmtime)
    print(f"✓ Found pickle file: {pickle_file}")
    print(f"  Last modified: {time.ctime(os.path.getmtime(pickle_file))}")
else:
    print("\n❌ ERROR: No processed_data.pkl file found!")
    print("\n💡 Solution:")
    print("   Run notebook 1_data_processing.ipynb first to generate processed_data.pkl")
    raise FileNotFoundError(f"Run Part 1 first to create processed_data.pkl")

with open(pickle_file, 'rb') as f:
    data_package = pickle.load(f)

all_data = data_package['all_data']
config_info = data_package['config_info']
paths = data_package['paths']

POSITION_MAP = config_info['POSITION_MAP']
VALUE_MAPPINGS = config_info['VALUE_MAPPINGS']
TURBULENCE_ORDER = config_info['TURBULENCE_ORDER']

base_path = paths['base_path']
output_dir = paths['output_dir']

print(f"✓ Data loaded: {len(all_data)} configurations")
print(f"✓ Output directory: {output_dir}")

✓ Found pickle file: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\processed_data.pkl
  Last modified: Sun Dec  7 15:04:57 2025
✓ Data loaded: 16 configurations
✓ Output directory: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG


## User Settings

In [11]:
# Convergence analysis settings
RUN_CONVERGENCE_ANALYSIS = True
CONVERGENCE_MAX_TRIM = 0.8
CONVERGENCE_NUM_TESTS = 20
num_iterations = 150

# Coefficient parameters
SPAN = 0.85344
CHORD = 0.1
AIR_DENSITY = 1.225
VELOCITY = 24.38

REFERENCE_AREA = SPAN * CHORD
DYNAMIC_PRESSURE = 0.5 * AIR_DENSITY * VELOCITY**2
Q_times_A = DYNAMIC_PRESSURE * REFERENCE_AREA

print(f"Settings: Max trim={CONVERGENCE_MAX_TRIM}, Tests={CONVERGENCE_NUM_TESTS}")
print(f"Q × A = {Q_times_A:.4f}")

Settings: Max trim=0.8, Tests=20
Q × A = 31.0704


## Helper Functions

In [12]:
def extract_aoa_number(aoa_string):
    try:
        return int(aoa_string.split('_')[1])
    except:
        return 0

## Convergence Analysis

In [13]:
# ==================== CONVERGENCE ANALYSIS TOOL ====================
# Inspired by GUI_Plotter_v12.txt methodology
# Tests different amounts of initial data removal to find optimal convergence window

def analyze_convergence(data_array, min_trim=0, max_trim=0.5, num_tests=10):
    """
    Analyze how statistics change when trimming different amounts of initial data.
    
    Parameters:
    - data_array: numpy array of lift or drag values
    - min_trim: minimum fraction of data to trim from beginning (0 to 1)
    - max_trim: maximum fraction of data to trim from beginning (0 to 1)
    - num_tests: number of trim amounts to test
    
    Returns:
    - Dictionary with trim fractions, means, std devs, and COVs
    """
    total_iterations = len(data_array)
    trim_fractions = np.linspace(min_trim, max_trim, num_tests)
    
    results = {
        'iterations_removed': [],
        'iterations_used': [],
        'mean': [],
        'std_dev': [],
        'cov': []
    }
    
    for trim_frac in trim_fractions:
        iterations_to_remove = int(total_iterations * trim_frac)
        trimmed_data = data_array[iterations_to_remove:]
        
        if len(trimmed_data) > 0:
            mean_val = np.mean(trimmed_data)
            std_val = np.std(trimmed_data)
            cov_val = (std_val / mean_val * 100) if mean_val != 0 else 0
            
            results['iterations_removed'].append(iterations_to_remove)
            results['iterations_used'].append(len(trimmed_data))
            results['mean'].append(mean_val)
            results['std_dev'].append(std_val)
            results['cov'].append(cov_val)
    
    return results

def plot_convergence_analysis(config, aoa, lift_data, drag_data, output_dir):
    """
    Create convergence analysis plots showing how statistics change with data trimming.
    """
    # Analyze both lift and drag using configured parameters
    lift_results = analyze_convergence(np.array(lift_data), min_trim=0, max_trim=CONVERGENCE_MAX_TRIM, num_tests=CONVERGENCE_NUM_TESTS)
    drag_results = analyze_convergence(np.array(drag_data), min_trim=0, max_trim=CONVERGENCE_MAX_TRIM, num_tests=CONVERGENCE_NUM_TESTS)
    
    # Create figure with subplots
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Lift Mean vs Iterations Removed
    ax1.plot(lift_results['iterations_removed'], lift_results['mean'], 'o-', linewidth=2, markersize=8, color='#1f77b4')
    ax1.set_xlabel('Iterations Removed from Start')
    ax1.set_ylabel('Lift Mean (N)')
    ax1.set_title(f'Lift Mean Convergence\n{config} - {aoa}', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Lift COV vs Iterations Removed
    ax2.plot(lift_results['iterations_removed'], lift_results['cov'], 'o-', linewidth=2, markersize=8, color='#ff7f0e')
    ax2.set_xlabel('Iterations Removed from Start')
    ax2.set_ylabel('Lift COV (%)')
    ax2.set_title(f'Lift COV Convergence\n{config} - {aoa}', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Highlight minimum COV point for lift
    min_cov_idx = np.argmin(lift_results['cov'])
    ax2.axvline(x=lift_results['iterations_removed'][min_cov_idx], color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax2.text(lift_results['iterations_removed'][min_cov_idx], max(lift_results['cov']), 
             f"  Min COV\n  Remove: {lift_results['iterations_removed'][min_cov_idx]}\n  Use: {lift_results['iterations_used'][min_cov_idx]}", 
             color='red', fontweight='bold', fontsize=13)
    
    # Plot 3: Drag Mean vs Iterations Removed
    ax3.plot(drag_results['iterations_removed'], drag_results['mean'], 'o-', linewidth=2, markersize=8, color='#2ca02c')
    ax3.set_xlabel('Iterations Removed from Start')
    ax3.set_ylabel('Drag Mean (N)')
    ax3.set_title(f'Drag Mean Convergence\n{config} - {aoa}', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Drag COV vs Iterations Removed
    ax4.plot(drag_results['iterations_removed'], drag_results['cov'], 'o-', linewidth=2, markersize=8, color='#d62728')
    ax4.set_xlabel('Iterations Removed from Start')
    ax4.set_ylabel('Drag COV (%)')
    ax4.set_title(f'Drag COV Convergence\n{config} - {aoa}', fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    # Highlight minimum COV point for drag
    min_cov_idx = np.argmin(drag_results['cov'])
    ax4.axvline(x=drag_results['iterations_removed'][min_cov_idx], color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax4.text(drag_results['iterations_removed'][min_cov_idx], max(drag_results['cov']), 
             f"  Min COV\n  Remove: {drag_results['iterations_removed'][min_cov_idx]}\n  Use: {drag_results['iterations_used'][min_cov_idx]}", 
             color='red', fontweight='bold', fontsize=13)
    
    plt.tight_layout()
    
    # Save convergence analysis plot
    convergence_dir = os.path.join(output_dir, "convergence_analysis")
    os.makedirs(convergence_dir, exist_ok=True)
    
    plot_file = os.path.join(convergence_dir, f"convergence_{config}.png")
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    return lift_results, drag_results, plot_file

# ==================== RUN CONVERGENCE ANALYSIS ====================
# Check if convergence analysis is enabled in configuration section above

if RUN_CONVERGENCE_ANALYSIS:
    print("\n" + "=" * 80)
    print("RUNNING CONVERGENCE ANALYSIS")
    print("=" * 80)
    
    # Create convergence analysis directory
    convergence_dir = os.path.join(output_dir, "convergence_analysis")
    os.makedirs(convergence_dir, exist_ok=True)
    
    # Run convergence analysis on all configurations
    convergence_results = {}
    
    # Create text file for convergence data
    convergence_text_file = os.path.join(convergence_dir, "Convergence_Analysis_Results.txt")
    
    with open(convergence_text_file, 'w') as f:
        f.write("=" * 120 + "\n")
        f.write("CONVERGENCE ANALYSIS RESULTS\n")
        f.write("=" * 120 + "\n\n")
        
        for (config, aoa), data in all_data.items():
            print(f"\nAnalyzing: {config} - {aoa}")
            
            lift_results, drag_results, plot_path = plot_convergence_analysis(
                config, aoa,
                data['lift'],
                data['drag'],
                output_dir
            )
            
            convergence_results[(config, aoa)] = {
                'lift': lift_results,
                'drag': drag_results,
                'plot': plot_path
            }
            
            # Print optimization recommendations
            lift_min_cov_idx = np.argmin(lift_results['cov'])
            drag_min_cov_idx = np.argmin(drag_results['cov'])
            
            print(f"  ✓ Plot saved: {plot_path}")
            print(f"  ✓ Lift - Min COV at {lift_results['iterations_removed'][lift_min_cov_idx]} iterations removed (use {lift_results['iterations_used'][lift_min_cov_idx]})")
            print(f"  ✓ Drag - Min COV at {drag_results['iterations_removed'][drag_min_cov_idx]} iterations removed (use {drag_results['iterations_used'][drag_min_cov_idx]})")
            
            # Write to text file
            f.write(f"Configuration: {config}\n")
            f.write(f"Angle of Attack: {aoa}\n")
            f.write(f"Total Iterations: {len(data['lift'])}\n")
            f.write("-" * 120 + "\n\n")
            
            # Write LIFT table
            f.write("LIFT CONVERGENCE:\n")
            f.write(f"{'Iterations_Removed':<20} {'Iterations_Used':<20} {'Mean':<15} {'StdDev':<15} {'COV(%)':<10}\n")
            f.write("-" * 120 + "\n")
            for i in range(len(lift_results['iterations_removed'])):
                marker = " <-- MIN COV" if i == lift_min_cov_idx else ""
                f.write(f"{lift_results['iterations_removed'][i]:<20} "
                       f"{lift_results['iterations_used'][i]:<20} "
                       f"{lift_results['mean'][i]:<15.6f} "
                       f"{lift_results['std_dev'][i]:<15.6f} "
                       f"{lift_results['cov'][i]:<10.2f}{marker}\n")
            
            f.write("\n")
            
            # Write DRAG table
            f.write("DRAG CONVERGENCE:\n")
            f.write(f"{'Iterations_Removed':<20} {'Iterations_Used':<20} {'Mean':<15} {'StdDev':<15} {'COV(%)':<10}\n")
            f.write("-" * 120 + "\n")
            for i in range(len(drag_results['iterations_removed'])):
                marker = " <-- MIN COV" if i == drag_min_cov_idx else ""
                f.write(f"{drag_results['iterations_removed'][i]:<20} "
                       f"{drag_results['iterations_used'][i]:<20} "
                       f"{drag_results['mean'][i]:<15.6f} "
                       f"{drag_results['std_dev'][i]:<15.6f} "
                       f"{drag_results['cov'][i]:<10.2f}{marker}\n")
            
            f.write("\n" + "=" * 120 + "\n\n")
    
    print("\n" + "=" * 80)
    print("CONVERGENCE ANALYSIS COMPLETE")
    print(f"✓ Total configurations analyzed: {len(convergence_results)}")
    print(f"✓ Plots saved to: {os.path.join(output_dir, 'convergence_analysis')}")
    print(f"✓ Text file saved: {convergence_text_file}")
    print("=" * 80)
    
else:
    print("\n✓ Convergence analysis skipped (set RUN_CONVERGENCE_ANALYSIS = True to enable)")


RUNNING CONVERGENCE ANALYSIS

Analyzing: 3.1.1.NG.10 - AoA_10
  ✓ Plot saved: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\convergence_analysis\convergence_3.1.1.NG.10.png
  ✓ Lift - Min COV at 960 iterations removed (use 240)
  ✓ Drag - Min COV at 909 iterations removed (use 291)

Analyzing: 3.1.1.NG.11 - AoA_11
  ✓ Plot saved: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\convergence_analysis\convergence_3.1.1.NG.10.png
  ✓ Lift - Min COV at 960 iterations removed (use 240)
  ✓ Drag - Min COV at 909 iterations removed (use 291)

Analyzing: 3.1.1.NG.11 - AoA_11
  ✓ Plot saved: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\convergence_analysis\convergence_3.1.1.NG.11.png
  ✓ Lift - Min COV at 353 iterations removed (use 847)
  ✓ Drag - Min COV at 353 iterations removed (use 847)

Analyzing: 3.1.1.NG.12 - AoA_1

## Save Convergence Results

In [14]:
if RUN_CONVERGENCE_ANALYSIS and 'convergence_results' in dir():
    conv_pickle = os.path.join(output_dir, 'convergence_results.pkl')
    with open(conv_pickle, 'wb') as f:
        pickle.dump({
            'convergence_results': convergence_results,
            'num_iterations': num_iterations
        }, f)
    print(f"✓ Convergence results saved to: {conv_pickle}")
    print("  Run notebook 3_excel_outputs.ipynb to use these optimized trim values")
else:
    print("⚠ No convergence results to save")

✓ Convergence results saved to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\convergence_results.pkl
  Run notebook 3_excel_outputs.ipynb to use these optimized trim values


## Export Optimized Data to Text Files

In [15]:
if RUN_CONVERGENCE_ANALYSIS and 'convergence_results' in dir():
    # Create post-processed data directory
    postprocessed_dir = os.path.join(output_dir, "convergence_analysis", "optimized_data")
    os.makedirs(postprocessed_dir, exist_ok=True)
    
    print("\n" + "=" * 80)
    print("EXPORTING OPTIMIZED DATA")
    print("=" * 80)
    
    # Export optimized data for each configuration
    for (config, aoa), conv_data in convergence_results.items():
        data = all_data[(config, aoa)]
        
        # Get optimal trim values
        lift_min_cov_idx = np.argmin(conv_data['lift']['cov'])
        drag_min_cov_idx = np.argmin(conv_data['drag']['cov'])
        
        optimal_lift_trim = conv_data['lift']['iterations_removed'][lift_min_cov_idx]
        optimal_drag_trim = conv_data['drag']['iterations_removed'][drag_min_cov_idx]
        optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
        
        # Get optimized data
        optimized_lift = data['lift'][optimal_trim:]
        optimized_drag = data['drag'][optimal_trim:]
        
        # Export optimized lift data
        lift_file = os.path.join(postprocessed_dir, f"{config}_lift_optimized.txt")
        with open(lift_file, 'w') as f:
            f.write(f"# Optimized Lift Data for {config}\n")
            f.write(f"# Trimmed {optimal_trim} iterations from start\n")
            f.write(f"# Using {len(optimized_lift)} iterations\n")
            f.write(f"# Mean: {np.mean(optimized_lift):.6f} N\n")
            f.write(f"# Std Dev: {np.std(optimized_lift):.6f} N\n")
            f.write(f"# COV: {(np.std(optimized_lift)/np.mean(optimized_lift)*100):.2f} %\n")
            f.write("#\n")
            for value in optimized_lift:
                f.write(f"{value}\n")
        
        # Export optimized drag data
        drag_file = os.path.join(postprocessed_dir, f"{config}_drag_optimized.txt")
        with open(drag_file, 'w') as f:
            f.write(f"# Optimized Drag Data for {config}\n")
            f.write(f"# Trimmed {optimal_trim} iterations from start\n")
            f.write(f"# Using {len(optimized_drag)} iterations\n")
            f.write(f"# Mean: {np.mean(optimized_drag):.6f} N\n")
            f.write(f"# Std Dev: {np.std(optimized_drag):.6f} N\n")
            f.write(f"# COV: {(np.std(optimized_drag)/np.mean(optimized_drag)*100):.2f} %\n")
            f.write("#\n")
            for value in optimized_drag:
                f.write(f"{value}\n")
    
    print(f"\n✓ Optimized data exported to: {postprocessed_dir}")
    print(f"✓ Total files created: {len(convergence_results) * 2} ({len(convergence_results)} configs × 2 files)")
    print("=" * 80)
else:
    print("\n⚠ Convergence analysis not run - no optimized data to export")


EXPORTING OPTIMIZED DATA

✓ Optimized data exported to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\convergence_analysis\optimized_data
✓ Total files created: 32 (16 configs × 2 files)

✓ Optimized data exported to: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\convergence_analysis\optimized_data
✓ Total files created: 32 (16 configs × 2 files)


## Coefficient Graphs (C_L and C_D vs AoA)

In [16]:
# ==================== CALCULATE COEFFICIENTS ====================
# Calculate C_L and C_D for each configuration

# Initialize convergence_results if not already defined (when RUN_CONVERGENCE_ANALYSIS is False)
if 'convergence_results' not in dir():
    convergence_results = {}

coefficient_data = {}

for (config, aoa), data in all_data.items():
    # Get optimized data if convergence analysis was run, otherwise use fixed iterations
    if RUN_CONVERGENCE_ANALYSIS and (config, aoa) in convergence_results:
        conv = convergence_results[(config, aoa)]
        lift_min_cov_idx = np.argmin(conv['lift']['cov'])
        drag_min_cov_idx = np.argmin(conv['drag']['cov'])
        
        optimal_lift_trim = conv['lift']['iterations_removed'][lift_min_cov_idx]
        optimal_drag_trim = conv['drag']['iterations_removed'][drag_min_cov_idx]
        optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
        
        lift_values = data['lift'][optimal_trim:]
        drag_values = data['drag'][optimal_trim:]
    else:
        # Use fixed iterations approach
        lift_values = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_values = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate mean forces
    lift_mean = np.mean(lift_values) if lift_values else 0
    drag_mean = np.mean(drag_values) if drag_values else 0
    
    # Calculate coefficients
    C_L = lift_mean / Q_times_A if Q_times_A != 0 else 0
    C_D = drag_mean / Q_times_A if Q_times_A != 0 else 0
    
    # Calculate standard deviations and COV for coefficients
    lift_std = np.std(lift_values) if lift_values else 0
    drag_std = np.std(drag_values) if drag_values else 0
    
    C_L_std = lift_std / Q_times_A if Q_times_A != 0 else 0
    C_D_std = drag_std / Q_times_A if Q_times_A != 0 else 0
    
    C_L_cov = (C_L_std / C_L * 100) if C_L != 0 else 0
    C_D_cov = (C_D_std / C_D * 100) if C_D != 0 else 0
    
    # Extract AoA number
    aoa_degrees = extract_aoa_number(aoa)
    
    # Extract config parts for filtering
    config_parts = config.split('.')
    
    # Get version number from config
    version_num = config_parts[POSITION_MAP['version']] if len(config_parts) > POSITION_MAP['version'] else 'unknown'
    
    # Store coefficient data
    coefficient_data[(config, aoa)] = {
        'config': config,
        'config_parts': config_parts,
        'version_num': version_num,
        'aoa_degrees': aoa_degrees,
        'C_L': C_L,
        'C_D': C_D,
        'C_L_std': C_L_std,
        'C_D_std': C_D_std,
        'C_L_cov': C_L_cov,
        'C_D_cov': C_D_cov,
        'turbulence_model': data['turbulence_model'],
        'geometry': data['geometry'],
        'mesh': data['mesh'],
        'grid': data['grid']
    }

print(f"\n✓ Coefficients calculated for {len(coefficient_data)} configurations")

# ==================== CREATE C_L vs AoA and C_D vs AoA GRAPHS ====================
# Create folder structure: coefficient_graphs / turbulence_model / config

# Get all unique base configurations (without AoA - first 4 parts)
unique_base_configs = set()
for (config, aoa), coeff in coefficient_data.items():
    parts = config.split('.')
    if len(parts) >= 4:
        base_config = '.'.join(parts[:4])  # e.g., "4.3.1.2"
        unique_base_configs.add(base_config)

# Group configurations by turbulence model
configs_by_turbulence = defaultdict(list)
for base_config in unique_base_configs:
    parts = base_config.split('.')
    turb_num = parts[POSITION_MAP['turbulence']] if len(parts) > POSITION_MAP['turbulence'] else 'unknown'
    turb_name = VALUE_MAPPINGS['turbulence'].get(int(turb_num) if turb_num.isdigit() else turb_num, f"Turbulence_{turb_num}")
    configs_by_turbulence[turb_name].append(base_config)

print(f"\n✓ Found {len(unique_base_configs)} unique configurations organized by turbulence model:")
for turb_name in sorted(configs_by_turbulence.keys()):
    print(f"\n  {turb_name}/")
    for cfg in sorted(configs_by_turbulence[turb_name]):
        print(f"    - {cfg}/")

# Define colors and markers for different turbulence models
model_styles = {
    'SST': {'color': '#1f77b4', 'marker': 'o', 'label': 'SST'},
    'RNG': {'color': '#ff7f0e', 'marker': 's', 'label': 'RNG k-ε'},
    'RSM': {'color': '#2ca02c', 'marker': '^', 'label': 'RSM'},
    'k-epsilon': {'color': '#d62728', 'marker': 'D', 'label': 'k-ε'}
}

# Create graphs organized by turbulence model
for turb_name in sorted(configs_by_turbulence.keys()):
    print(f"\n{'='*70}")
    print(f"TURBULENCE MODEL: {turb_name}")
    print(f"{'='*70}")
    
    for base_config in sorted(configs_by_turbulence[turb_name]):
        # Create directory: coefficient_graphs / turbulence_model / config
        config_graphs_dir = os.path.join(output_dir, "coefficient_graphs", turb_name, base_config)
        os.makedirs(config_graphs_dir, exist_ok=True)
        
        # Get descriptive info for this config
        parts = base_config.split('.')
        geometry_name = VALUE_MAPPINGS['geometry'].get(int(parts[0]) if parts[0].isdigit() else parts[0], f"Geometry_{parts[0]}")
        mesh_name = VALUE_MAPPINGS['mesh'].get(int(parts[1]) if parts[1].isdigit() else parts[1], f"Mesh_{parts[1]}")
        version_name = VALUE_MAPPINGS['version'].get(int(parts[3]) if parts[3].isdigit() else parts[3], f"Version_{parts[3]}")
        
        config_description = f"{geometry_name}, {mesh_name}, {turb_name}, {version_name}"
        
        print(f"\n  Creating graphs for {base_config}")
        print(f"    ({config_description})")
        
        # Filter coefficient data for this base configuration
        config_coeff_data = {}
        for (config, aoa), coeff in coefficient_data.items():
            config_parts = config.split('.')
            if len(config_parts) >= 4:
                this_base = '.'.join(config_parts[:4])
                if this_base == base_config:
                    config_coeff_data[(config, aoa)] = coeff
        
        if not config_coeff_data:
            print(f"    ⚠ No data found for {base_config}")
            continue
        
        # Organize data by AoA for plotting
        aoa_list = []
        C_L_list = []
        C_D_list = []
        C_L_std_list = []
        C_D_std_list = []
        
        for (config, aoa), coeff in config_coeff_data.items():
            aoa_list.append(coeff['aoa_degrees'])
            C_L_list.append(coeff['C_L'])
            C_D_list.append(coeff['C_D'])
            C_L_std_list.append(coeff['C_L_std'])
            C_D_std_list.append(coeff['C_D_std'])
        
        # Sort by AoA
        combined = list(zip(aoa_list, C_L_list, C_D_list, C_L_std_list, C_D_std_list))
        combined.sort(key=lambda x: x[0])
        
        aoa_vals = np.array([x[0] for x in combined])
        C_L_vals = np.array([x[1] for x in combined])
        C_D_vals = np.array([x[2] for x in combined])
        C_L_std_vals = np.array([x[3] for x in combined])
        C_D_std_vals = np.array([x[4] for x in combined])
        
        # Get style based on turbulence model
        style = model_styles.get(turb_name, {'color': '#1f77b4', 'marker': 'o', 'label': turb_name})
        
        # ==================== PLOT 1: C_L vs AoA ====================
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.errorbar(aoa_vals, C_L_vals, yerr=C_L_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax.set_xlabel('Angle of Attack (degrees)', fontsize=18, fontweight='bold')
        ax.set_ylabel('Lift Coefficient ($C_L$)', fontsize=18, fontweight='bold')
        ax.set_title(f'Lift Coefficient vs Angle of Attack\n{base_config}: {config_description}', fontsize=18, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=16, loc='best', framealpha=0.9)
        ax.tick_params(labelsize=15)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "C_L_vs_AoA.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # ==================== PLOT 2: C_D vs AoA ====================
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.errorbar(aoa_vals, C_D_vals, yerr=C_D_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax.set_xlabel('Angle of Attack (degrees)', fontsize=18, fontweight='bold')
        ax.set_ylabel('Drag Coefficient ($C_D$)', fontsize=18, fontweight='bold')
        ax.set_title(f'Drag Coefficient vs Angle of Attack\n{base_config}: {config_description}', fontsize=18, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=16, loc='best', framealpha=0.9)
        ax.tick_params(labelsize=15)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "C_D_vs_AoA.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # ==================== PLOT 3: C_L vs C_D (Drag Polar) ====================
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.errorbar(C_D_vals, C_L_vals, xerr=C_D_std_vals, yerr=C_L_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        # Add AoA annotations at each point
        for i, aoa in enumerate(aoa_vals):
            ax.annotate(f"{int(aoa)}°", 
                       (C_D_vals[i], C_L_vals[i]),
                       textcoords="offset points", xytext=(8, 5),
                       fontsize=14, fontweight='bold', color=style['color'])
        
        ax.set_xlabel('Drag Coefficient ($C_D$)', fontsize=18, fontweight='bold')
        ax.set_ylabel('Lift Coefficient ($C_L$)', fontsize=18, fontweight='bold')
        ax.set_title(f'Drag Polar ($C_L$ vs $C_D$)\n{base_config}: {config_description}', fontsize=18, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=16, loc='best', framealpha=0.9)
        ax.tick_params(labelsize=15)
        
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "Drag_Polar.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        # ==================== PLOT 4: Combined C_L and C_D vs AoA ====================
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
        
        # Left plot: C_L vs AoA
        ax1.errorbar(aoa_vals, C_L_vals, yerr=C_L_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax1.set_xlabel('Angle of Attack (degrees)', fontsize=18, fontweight='bold')
        ax1.set_ylabel('Lift Coefficient ($C_L$)', fontsize=18, fontweight='bold')
        ax1.set_title(f'Lift Coefficient vs AoA\n{base_config}', fontsize=18, fontweight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--')
        ax1.legend(fontsize=15, loc='best', framealpha=0.9)
        ax1.tick_params(labelsize=15)
        
        # Right plot: C_D vs AoA
        ax2.errorbar(aoa_vals, C_D_vals, yerr=C_D_std_vals,
                    marker=style['marker'], markersize=10, linewidth=2.5, capsize=5,
                    color=style['color'], label=f'{turb_name}', alpha=0.9)
        
        ax2.set_xlabel('Angle of Attack (degrees)', fontsize=18, fontweight='bold')
        ax2.set_ylabel('Drag Coefficient ($C_D$)', fontsize=18, fontweight='bold')
        ax2.set_title(f'Drag Coefficient vs AoA\n{base_config}', fontsize=18, fontweight='bold')
        ax2.grid(True, alpha=0.3, linestyle='--')
        ax2.legend(fontsize=15, loc='best', framealpha=0.9)
        ax2.tick_params(labelsize=15)
        
        plt.suptitle(config_description, fontsize=16, y=0.02)
        plt.tight_layout()
        plt.savefig(os.path.join(config_graphs_dir, "C_L_C_D_Combined.png"), dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"    ✓ All 4 graphs saved to: {turb_name}/{base_config}/")

print("\n" + "=" * 80)
print("COEFFICIENT GRAPHS COMPLETE")
print(f"✓ Folder structure: coefficient_graphs / turbulence_model / config")
print(f"✓ Location: {os.path.join(output_dir, 'coefficient_graphs')}")
print(f"\nFolder structure:")
for turb_name in sorted(configs_by_turbulence.keys()):
    print(f"  {turb_name}/")
    for cfg in sorted(configs_by_turbulence[turb_name]):
        print(f"    └── {cfg}/")
print("\n✓ Each config folder contains: C_L_vs_AoA.png, C_D_vs_AoA.png, Drag_Polar.png, C_L_C_D_Combined.png")


✓ Coefficients calculated for 16 configurations

✓ Found 1 unique configurations organized by turbulence model:

  SST/
    - 3.1.1.NG/

TURBULENCE MODEL: SST

  Creating graphs for 3.1.1.NG
    (2414_006_002, Coarse, SST, Version_NG)


    ✓ All 4 graphs saved to: SST/3.1.1.NG/

COEFFICIENT GRAPHS COMPLETE
✓ Folder structure: coefficient_graphs / turbulence_model / config
✓ Location: C:\Users\<YOUR_USERNAME>\OneDrive\Documents\Thesis\NACA_2414_2D\Fleunt\Directories\2414_006_005\3.1.1.NG\coefficient_graphs

Folder structure:
  SST/
    └── 3.1.1.NG/

✓ Each config folder contains: C_L_vs_AoA.png, C_D_vs_AoA.png, Drag_Polar.png, C_L_C_D_Combined.png
